In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [17]:
BOLD = "\033[1m"
CYAN = "\033[36m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
MAG = "\033[95m"
RESET = "\033[0m"

In [2]:
import yaml
from src.data.loader import load_dataset
from src.data.validation import validate_dataset

In [3]:
BENCHMARK_CONFIG = PROJECT_ROOT / "configs" / "benchmark.yaml"
TARGET_CONFIG = PROJECT_ROOT / "configs" / "benchmark_targets.yaml"

In [5]:
with open(BENCHMARK_CONFIG, "r", encoding="utf-8") as f:
    benchmark = yaml.safe_load(f)["benchmark"]
print(f"Successfully loaded {BOLD}{YELLOW}{BENCHMARK_CONFIG.relative_to(PROJECT_ROOT).as_posix()}")

Successfully loaded configs/benchmark.yaml


In [6]:
with open(TARGET_CONFIG, "r", encoding="utf-8") as f:
    targets = yaml.safe_load(f)["targets"]
print(f"Successfully loaded {BOLD}{YELLOW}{TARGET_CONFIG.relative_to(PROJECT_ROOT).as_posix()}")

Successfully loaded configs/benchmark_targets.yaml


In [19]:
context_length = benchmark["context_length"]
max_horizon = max(benchmark["prediction_lengths"])
required_length = context_length + max_horizon

In [21]:
for dataset_name in benchmark["datasets"]:
    dataset = load_dataset(dataset_name)
    report = validate_dataset(dataset)
    
    dashes = "-" * 50
    print(f"----- {BOLD}{YELLOW}{dataset_name:12}{RESET}{dashes}")
    
    print(f"  + Observations:    {CYAN}{dataset.n_observations}{RESET}")
    print(f"  + Variates:        {CYAN}{dataset.n_variates}{RESET}")
    print(f"  + Frequency:       {CYAN}{dataset.frequency}{RESET}")
    print(f"  + Seasonal period: {CYAN}{dataset.seasonal_period}{RESET}")
    print(f"  + Missing rate:    {CYAN}{report['missing_rate']:.4f}{RESET}")
    print(f"  + Targets:         {MAG}{targets[dataset_name]}{RESET}")
    
    if dataset.n_observations < required_length:
        raise ValueError(f"  *** {BOLD}{YELLOW}{dataset_name}{RESET} has only {BOLD}{CYAN}{dataset.n_observations}{RESET} observations. At least {BOLD}{CYAN}{required_length}{RESET} are required.")
    
    for target in targets[dataset_name]:
        if target not in dataset.values.columns:
            raise ValueError(f"Target {BOLD}{MAG}{target}{RESET} does not exist in {BOLD}{YELLOW}{dataset_name}")
        
    print()
    
print("All benchmark datasets passed validation.")

----- ETTh1       --------------------------------------------------
  + Observations:    17420
  + Variates:        7
  + Frequency:       H
  + Seasonal period: 24
  + Missing rate:    0.0000
  + Targets:         ['OT', 'HULL', 'MULL']

----- Weather     --------------------------------------------------
  + Observations:    52695
  + Variates:        21
  + Frequency:       None
  + Seasonal period: 144
  + Missing rate:    0.0000
  + Targets:         ['OT', 'rh (%)', 'SWDR (W/m�)']

----- Electricity --------------------------------------------------
  + Observations:    26304
  + Variates:        321
  + Frequency:       H
  + Seasonal period: 24
  + Missing rate:    0.0000
  + Targets:         ['146', '22', '163']

----- Traffic     --------------------------------------------------
  + Observations:    17544
  + Variates:        862
  + Frequency:       H
  + Seasonal period: 24
  + Missing rate:    0.0000
  + Targets:         ['T683', 'T472', 'T855']

----- Exchange    --------